# Observability Demo

End-to-end walkthrough of the Sprint 0.3 observability surface:

1. **Structured logging** — `configure_logging`, `@log_call`, `log_dataframe`.
2. **Per-run lineage** — `run_context` + `attach_artifact` for non-model work.
3. **MLflow** — shared tracking URI, experiment setup, dataframe/econ metric helpers.
4. **Metrics** — `economic_cost`, `precision_recall_at_k`, `recall_at_fpr`, `compute_psi`.

For the operator perspective — reading logs, tracing a lineage issue end-to-end, log-level conventions — see [docs/OBSERVABILITY.md](../docs/OBSERVABILITY.md).

## 1. Structured logging

`configure_logging(pipeline_name)` returns a `run_id` and wires both a JSON stdout handler and a text-file handler under `logs/{pipeline_name}/{run_id}.log`. `@log_call` instruments individual functions.

In [1]:
from __future__ import annotations

import matplotlib

matplotlib.use("Agg")

from fraud_engine.utils.logging import configure_logging, get_logger, log_call

run_id = configure_logging("observability-demo")
logger = get_logger(__name__)
logger.info("notebook.start", run_id=run_id)


@log_call
def square(x: int) -> int:
    """Square the input; used only to demonstrate `@log_call`."""
    return x * x


square(7)

{"run_id": "6a1dcd4a4eec44bbbd49f20954f97226", "event": "notebook.start", "pipeline": "observability-demo", "logger": "__main__", "level": "info", "timestamp": "2026-04-28T01:35:06.623483Z"}


{"arg_0": {"type": "int", "value": 7}, "event": "square.start", "run_id": "6a1dcd4a4eec44bbbd49f20954f97226", "pipeline": "observability-demo", "logger": "__main__", "level": "info", "timestamp": "2026-04-28T01:35:06.624871Z"}


{"duration_ms": 0.001, "result": {"type": "int", "value": 49}, "event": "square.done", "run_id": "6a1dcd4a4eec44bbbd49f20954f97226", "pipeline": "observability-demo", "logger": "__main__", "level": "info", "timestamp": "2026-04-28T01:35:06.625725Z"}


49

## 2. `log_dataframe` — shape snapshots at stage boundaries

A synthetic DataFrame stands in here so the notebook runs in CI without the raw IEEE-CIS CSVs. In Sprint 1+, callers will pass `RawDataLoader().load_transactions().head(100)` or a feature-pipeline output.

In [2]:
import numpy as np
import pandas as pd

from fraud_engine.utils.logging import log_dataframe

rng = np.random.default_rng(0)
demo_df = pd.DataFrame(
    {
        "amount": rng.uniform(1.0, 500.0, 100).round(2),
        "currency": rng.choice(["USD", "EUR", "GBP"], 100),
        "is_fraud": rng.choice([0, 1], 100, p=[0.95, 0.05]).astype(int),
    }
)
log_dataframe(demo_df, name="demo-slice")
demo_df.head()

{"name": "demo-slice", "rows": 100, "cols": 3, "memory_mb": 0.0074, "dtypes": {"float64": 1, "object": 1, "int64": 1}, "n_missing": 0, "first_row_sha256": "f1924ad184a5d2a09400b67069b5f835bc230111005896cf15c28611a54c2c3d", "event": "dataframe.snapshot", "run_id": "6a1dcd4a4eec44bbbd49f20954f97226", "pipeline": "observability-demo", "logger": "fraud_engine.utils.logging", "level": "info", "timestamp": "2026-04-28T01:35:06.639884Z"}


,amount,currency,is_fraud
0,318.84,USD,0
1,135.62,EUR,0
2,21.45,USD,0
3,9.25,USD,0
4,406.82,GBP,0


## 3. `run_context` — persistent per-run directory + artifacts

`run_context` is for non-model work: feature builds, one-off scripts, data downloads. Model training uses MLflow (below). Both cross-reference via a shared `run_id`.

`capture_streams=False` here so the notebook's output remains visible; production scripts default to `True`.

In [3]:
import json

import matplotlib.pyplot as plt

from fraud_engine.utils.tracing import attach_artifact, run_context

with run_context(
    "observability-demo",
    metadata={"source": "notebook", "rows": int(len(demo_df))},
    capture_streams=False,
) as run:
    fig, ax = plt.subplots()
    ax.hist(demo_df["amount"], bins=20)
    ax.set_xlabel("amount (USD)")
    ax.set_ylabel("count")
    attach_artifact(run, fig, name="amount_histogram")
    plt.close(fig)

    attach_artifact(run, demo_df.head(10), name="demo_head")
    attach_artifact(
        run,
        {"n_rows": int(len(demo_df)), "run_id": run.run_id},
        name="summary",
    )

artifacts = sorted(p.name for p in run.artifacts_dir.iterdir())
payload = json.loads((run.run_dir / "run.json").read_text(encoding="utf-8"))
print("artifacts:", artifacts)
print("status  :", payload["status"])
print("run_id  :", run.run_id)

{"run_id": "da009f3ec5c34bf7b81704c16cc0af4a", "pipeline": "observability-demo", "run_dir": "/home/dchit/projects/fraud-detection-engine/logs/runs/da009f3ec5c34bf7b81704c16cc0af4a", "event": "run.start", "logger": "fraud_engine.utils.tracing", "level": "info", "timestamp": "2026-04-28T01:35:06.666281Z"}


{"run_id": "da009f3ec5c34bf7b81704c16cc0af4a", "duration_ms": 67.882, "event": "run.done", "pipeline": "observability-demo", "logger": "fraud_engine.utils.tracing", "level": "info", "timestamp": "2026-04-28T01:35:06.734179Z"}


artifacts: ['amount_histogram.png', 'demo_head.parquet', 'summary.json']
status  : success
run_id  : da009f3ec5c34bf7b81704c16cc0af4a


## 4. MLflow — shared tracking URI + model-family experiment

The tracking URI is redirected to a temp directory here so the notebook doesn't pollute the repo's `./mlruns`. In production, `docker-compose.dev.yml` runs the tracking server at `http://localhost:5000`.

In [4]:
import os
import tempfile

import mlflow

from fraud_engine.config.settings import get_settings
from fraud_engine.utils.mlflow_setup import (
    configure_mlflow,
    log_dataframe_stats,
    log_economic_metrics,
    setup_experiment,
)

tmp_mlruns = tempfile.mkdtemp(prefix="nb-mlflow-")
os.environ["MLFLOW_TRACKING_URI"] = f"file:{tmp_mlruns}"
get_settings.cache_clear()

configure_mlflow()
experiment_id = setup_experiment("observability-demo")
with mlflow.start_run(experiment_id=experiment_id, run_name=run.run_id):
    log_dataframe_stats(demo_df, prefix="demo")
    log_economic_metrics(fn_count=8, fp_count=2, tp_count=5, tn_count=85, fraud_cost=450.0, fp_cost=35.0, tp_cost=5.0)

print(f"tracking_uri   : {get_settings().mlflow_tracking_uri}")
print(f"experiment_id  : {experiment_id}")
print("In dev, open http://localhost:5000 after `make docker-up`.")

{"tracking_uri": "file:/tmp/nb-mlflow-bl3ue1l4", "event": "mlflow.tracking_uri_set", "run_id": "da009f3ec5c34bf7b81704c16cc0af4a", "pipeline": "observability-demo", "logger": "fraud_engine.utils.mlflow_setup", "level": "info", "timestamp": "2026-04-28T01:35:06.744962Z"}


{"name": "observability-demo", "experiment_id": "438820994035760708", "tracking_uri": "file:/tmp/nb-mlflow-bl3ue1l4", "event": "mlflow.experiment_ready", "run_id": "da009f3ec5c34bf7b81704c16cc0af4a", "pipeline": "observability-demo", "logger": "fraud_engine.utils.mlflow_setup", "level": "info", "timestamp": "2026-04-28T01:35:06.759936Z"}


/home/dchit/projects/fraud-detection-engine/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


tracking_uri   : file:/tmp/nb-mlflow-bl3ue1l4
experiment_id  : 438820994035760708
In dev, open http://localhost:5000 after `make docker-up`.


## 5. Metrics — the four numbers Sprint 4 & Sprint 6 both call

`economic_cost` is the Sprint 4 threshold-optimisation objective. `recall_at_fpr` targets the operational FPR budget. `compute_psi` is the Sprint 6 drift alarm. Defining them once here keeps evaluation and monitoring in lockstep.

In [5]:
from fraud_engine.utils.metrics import (
    compute_psi,
    economic_cost,
    precision_recall_at_k,
    recall_at_fpr,
)

y_true = np.array([1, 0, 1, 0, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 0, 0, 0, 1])
y_scores = np.array([0.9, 0.2, 0.8, 0.1, 0.05, 0.95, 0.15, 0.1, 0.7, 0.3])

cost = economic_cost(y_true, y_pred)
precision_at_k, recall_at_k = precision_recall_at_k(y_true, y_scores, k=0.4)
recall_budget = recall_at_fpr(y_true, y_scores, target_fpr=0.2)

baseline = rng.normal(0.0, 1.0, 2000)
drifted = rng.normal(0.5, 1.0, 2000)
psi = compute_psi(baseline, drifted)

print(f"economic_cost (defaults)   : ${cost['total_cost']:,.2f}")
print(f"precision@40% / recall@40% : {precision_at_k:.3f} / {recall_at_k:.3f}")
print(f"recall @ FPR=0.2           : {recall_budget:.3f}")
print(f"PSI (0.5σ mean shift)      : {psi:.4f}")

economic_cost (defaults)   : $980.00
precision@40% / recall@40% : 1.000 / 1.000
recall @ FPR=0.2           : 1.000
PSI (0.5σ mean shift)      : 0.2776


## 6. Reading run summaries — `jq`-style with `json.loads`

`configure_logging` writes two streams: a **JSON-per-line** stream on stdout (the format ELK / Loki / `jq` consume) and a **human-readable text mirror** at `logs/{pipeline}/{run_id}.log` for `tail -f`. The proximate JSON file in this repo is the per-run summary at `logs/runs/{run_id}/run.json`, written by `run_context` — that's what the cell below parses.

`jq` isn't shipped with this dev environment, so the cell uses stdlib `json.loads` and `json.dumps(indent=2)` to do the same filter+pretty-print. The full set of canonical jq recipes lives in [`docs/OBSERVABILITY.md` §2.2](../docs/OBSERVABILITY.md).

In [6]:
import json

# Cell 3's `run_context` wrote a per-run JSON summary at
# logs/runs/{run.run_id}/run.json. That's the proximate JSON file in
# this repo — the on-disk text mirror at logs/observability-demo/
# {run_id}.log is structlog's ConsoleRenderer output (human-readable
# for tail -f, not jq-parseable). The canonical JSON stream is on
# stdout, captured by ELK / Loki in prod.
run_json_path = run.run_dir / "run.json"
print(f"reading {run_json_path}")

# jq equivalents (run from the repo root in a shell with jq):
#   jq . logs/runs/<run_id>/run.json                        # full pretty-print
#   jq '.metadata' logs/runs/<run_id>/run.json              # just the metadata block
#   jq 'select(.status == "success")' logs/runs/<run_id>/run.json
record = json.loads(run_json_path.read_text(encoding="utf-8"))
print("\npretty-printed run summary:\n")
print(json.dumps(record, indent=2, sort_keys=True))

# Close out the structlog stream with one final event so the run has
# a clean open / close pair. `run_id` here is the configure_logging id
# from Cell 1; `run.run_id` is the run_context id from Cell 3.
logger.info("notebook.end", configure_logging_id=run_id, run_context_id=run.run_id)

reading /home/dchit/projects/fraud-detection-engine/logs/runs/da009f3ec5c34bf7b81704c16cc0af4a/run.json

pretty-printed run summary:

{
  "duration_ms": 67.882,
  "end_time": "2026-04-28 01:35:06.733940+00:00",
  "metadata": {
    "rows": 100,
    "source": "notebook"
  },
  "pipeline": "observability-demo",
  "run_id": "da009f3ec5c34bf7b81704c16cc0af4a",
  "start_time": "2026-04-28T01:35:06.666058+00:00",
  "status": "success"
}
{"configure_logging_id": "6a1dcd4a4eec44bbbd49f20954f97226", "run_context_id": "da009f3ec5c34bf7b81704c16cc0af4a", "event": "notebook.end", "run_id": "da009f3ec5c34bf7b81704c16cc0af4a", "pipeline": "observability-demo", "logger": "__main__", "level": "info", "timestamp": "2026-04-28T01:35:06.968939Z"}


## What to read next

- [`docs/OBSERVABILITY.md`](../docs/OBSERVABILITY.md) — the day-to-day operator guide (reading logs, tracing lineage, log-level table).
- [`src/fraud_engine/utils/tracing.py`](../src/fraud_engine/utils/tracing.py) — `run_context` internals.
- [`src/fraud_engine/utils/mlflow_setup.py`](../src/fraud_engine/utils/mlflow_setup.py) — MLflow helpers.
- [`src/fraud_engine/utils/metrics.py`](../src/fraud_engine/utils/metrics.py) — cost and drift formulas.

Every Sprint 1+ transformation wraps in `@log_call`; every one-off script wraps in `run_context(...)` so the lineage trail from raw CSVs to model predictions is unbroken.